# 03: 标准化 + 高可变基因选择（批次感知）

在 02 合并后的 counts 矩阵上执行标准化管线，为后续 04 降维与 05 聚类准备数据。

**标准化**消除文库大小差异——不同细胞的测序深度天然不同，不做标准化会导致
高深度细胞主导后续所有分析。**高可变基因（HVG）选择**决定下游分析使用哪些基因——
只保留携带细胞类型差异信息的高可变基因，大幅降噪并节省内存。
**batch-aware HVG** 是整合分析的关键 trick：在每个批次（source_dataset）内独立选 HVG，
取跨批次共识——防止某个数据集的技术噪声基因霸占 HVG 列表。

**本 notebook 新增能力**（v2 科研工作台升级）：
- 标准化双模：经典 `normalize_total + log1p` 或 Pearson residuals（Lause 2021）
- HVG Scalar-or-Sweep 双模：写单值直接跑，写列表自动对比不同参数组合的 HVG 重叠度
- 批次感知 HVG：`batch_key` 指定按哪个列独立选 HVG
- HVG 排除列表：自动排除线粒体/核糖体/血红蛋白基因，避免技术性基因驱动聚类
- 可选回归混杂变量（`regress_out`）与缩放（`scale`），默认关闭

**本 notebook 产出**：
- `adata.layers['counts']` — 原始 counts，为下游 scVI / scANVI / DESeq2 保留
- `adata.X` — 标准化后的表达矩阵（float32）
- `adata.var['highly_variable']` — HVG 布尔掩码
- `adata.var['hvg_{flavor}_{n}']` — 各参数组合的 HVG 掩码（sweep 模式）
- `adata.uns['normalize_v1']` — 标准化参数记录
- 03 checkpoint `.h5ad` 文件，供 04 嵌入使用

## 参数（唯一入口，四组）

本 stage 只需改这里的参数、读 QC 与诊断、确认后继续。四组分别是数据来源、科学参数、计算与开关、输出与运行标识。


In [ ]:
# === PARAMS ===
# === 组一：数据与版本（数据源）===
# 本组指定上游 02 checkpoint 的定位（run 目录 + run_id）；
# 改这里等于换输入数据，preflight 会校验其真实存在且 stage==02_merged。
UPSTREAM_RUN_ROOT = "results/runs"
UPSTREAM_RUN_ID = "02-merged-v1-run001"


### 科学参数（标准化 + HVG 选择）

以下每个参数都写了四要素注释（是什么/默认依据/调大调小影响/何时该改），改动会直接改变下游 PCA/聚类/批次校正结果。


In [ ]:
# === PARAMS ===
# === 组二：科学参数（标准化 + HVG 选择）===

# NORMALIZATION_METHOD：是什么=选标准化路线；
# 默认依据=standard(normalize_total+log1p) 是 scRNA 主流、稳健、下游工具默认假设；
# 调大调小=换 pearson_residuals(Lause 2021) 方差稳定化更强、对 dropout 更鲁棒但跳过 log1p 且与 seurat flavor 不兼容；
# 何时该改=高技术噪声/UMI 差异极大或需更强方差稳定化时试 pearson_residuals，并注意 HVG flavor 会自动切 seurat_v3。
# ⚠️ 注意：pearson_residuals 是实验性 API（sc.experimental.pp），scanpy 未来可能改动接口。
NORMALIZATION_METHOD = "standard"   # "standard" | "pearson_residuals"

# TARGET_SUM：是什么=每细胞缩放到的总计数目标（仅 standard 生效）；
# 默认依据=1e4（CP10K）是领域约定俗成；
# 调大调小=只是全局标度平移，log1p 后近似常数偏移，一般不影响相对结构；
# 何时该改=需与特定参考/图谱对齐标度时。
TARGET_SUM = 1e4                    # normalize_total 的 target（仅 standard 模式生效）

# N_TOP_GENES：是什么=选入 HVG 的基因数（单值或列表 sweep）；
# 默认依据=3000 兼顾胃全层 15-20 种细胞类型的稀有群体覆盖；
# 调大调小=调大纳入更多噪声但保稀有信号、调小更聚焦但可能丢稀有群体；
# 何时该改=细胞类型更少可降到 1500-2000，怀疑丢稀有群体或想 sweep 时写成列表。
N_TOP_GENES = 3000                  # 单值 | 列表如 [1500, 2000, 3000, 4000]

# HVG_FLAVOR：是什么=HVG 计算的 flavor，控制方差统计模型；
# 默认依据=seurat_v3（scanpy 推荐，内部自动从 layer="counts" 读取原始 counts，适合 UMI 数据）；
# 调大调小=seurat 用归一化后的 X（经典方法，小数据集可选）；
#           cell_ranger=Cell Ranger 风格，用归一化后的 X（复现 Cell Ranger 时用）；
#           seurat_v3=基于方差排名，对跨数据集整合更稳，pearson_residuals 必须用此项；
# 何时改=小数据集(<1000 cells)可试 seurat；需复现 Cell Ranger 用 cell_ranger；
#         pearson_residuals 模式会自动强制切换到 seurat_v3（无需手动修改）。
HVG_FLAVOR  = "seurat_v3"          # 单值 | 列表如 ["seurat", "seurat_v3"]

# BATCH_AWARE_HVG：是什么=是否按批次分别选 HVG 再合并；
# 默认依据=多数据集整合关键 trick，防单数据集技术噪声霸占 HVG；
# 调大调小=关闭会让大/噪声数据集主导 HVG；
# 何时该改=单一数据集或确认无批次结构时可关。
BATCH_AWARE_HVG = True

# HVG_BATCH_KEY：是什么=batch-aware 依据的 obs 列；
# 默认依据=source_dataset 对应数据来源；
# 何时该改=批次的真实来源是别的列（如样本/测序批）时。
HVG_BATCH_KEY   = "source_dataset"  # 按哪个列做 batch-aware

# EXCLUDE_MT_FROM_HVG：是什么=是否把线粒体基因排除出 HVG；
# 默认依据=线粒体基因反映技术/通用状态而非细胞身份，纳入会污染聚类轴；
# 调大调小=保留会让 PCA 被质量/污染信号主导；
# 何时该改=专门研究线粒体相关通路时才保留。
EXCLUDE_MT_FROM_HVG   = True        # 线粒体基因（MT-*）

# EXCLUDE_RIBO_FROM_HVG：是什么=是否把核糖体基因排除出 HVG；
# 默认依据=核糖体基因反映蛋白质合成活性而非细胞身份，纳入会污染聚类轴；
# 调大调小=保留会让 PCA 被通用转录活性信号主导；
# 何时该改=专门研究翻译调控时才保留。
EXCLUDE_RIBO_FROM_HVG = True        # 核糖体基因（RPS*/RPL*）

# EXCLUDE_HB_FROM_HVG：是什么=是否把血红蛋白基因排除出 HVG；
# 默认依据=血红蛋白基因反映红细胞污染而非细胞身份，纳入会污染聚类轴；
# 调大调小=保留会让 PCA 被污染信号主导；
# 何时该改=专门研究红细胞群体时才保留。
EXCLUDE_HB_FROM_HVG   = True        # 血红蛋白基因（HBA*/HBB*）

# CUSTOM_EXCLUDE_PATTERNS：是什么=额外排除正则；
# 默认依据=空=不额外排除；
# 何时该改=想排 IG/TCR 等克隆型基因（如 ["^IG[HKL]","^TR[ABGD]"]）时。
CUSTOM_EXCLUDE_PATTERNS = []        # 额外排除 pattern，如 ["^IG[HKL]", "^TR[ABGD]"]（免疫球蛋白/TCR）

# EXCLUDE_CELL_CYCLE_FROM_HVG：是什么=是否排除 Tirosh 2015 S+G2M 增殖基因；
# 默认依据=False 保留细胞周期信号；
# 何时该改=增殖信号主导 PCA、掩盖细胞身份时置 True。
EXCLUDE_CELL_CYCLE_FROM_HVG = False  # True=将 Tirosh 2015 S+G2M 基因加入排除列表（增殖信号主导 PCA 时启用）

# FORCED_INCLUDE_GENES：是什么=强制纳入 HVG 的关键基因；
# 默认依据=空；
# 调大调小=纳入越多越保关键生物学轴但可能引入低变异噪声；
# 何时该改=保证炎-癌转化 marker（如 CDX2/TFF3/OLFM4/LGR5）一定进入下游 PCA 空间时。
FORCED_INCLUDE_GENES = []           # PI 按需填入关键基因，如 ["CDX2", "TFF3", "GKN1", "MUC2", "OLFM4", "LGR5"]

# REGRESS_OUT：是什么=线性回归剔除的协变量；
# 默认依据=空（回归会 densify 矩阵、增内存与时间，且更推荐在 04 用 batch_key 处理）；
# 何时该改=PI 确认某混杂严重干扰 HVG/PCA 时，慎用。
REGRESS_OUT = []                    # 可选：["pct_counts_mt", "total_counts", "S_score", "G2M_score"]

# SCALE：是什么=是否对每基因 z-score 缩放；
# 默认依据=False（Harmony/scVI 不需要 scale，且 scale 会 densify）；
# 何时该改=PCA-only 管线可能需要。
SCALE = False

# MAX_SCALE_VALUE：是什么=scale 时的截断值；
# 默认依据=10 限制极端值影响；
# 何时该改=数据分布异常时调整。
MAX_SCALE_VALUE = 10

# N_PCS_SANITY：是什么=Sanity PCA 诊断的 PC 数（仅用于 elbow plot，不影响最终分析）；
# 默认依据=30（通常足够看到 elbow，捕捉主要方差来源）；
# 调大调小=调大覆盖更多 PC 但图更密；调小可加快速度；
# 何时改=数据集很小(<500 cells)时可降到 20，超大数据集维持 30 即可。
N_PCS_SANITY = 30

# N_PCS_FINAL：是什么=最终分析用的 PC 数（留空则需看完 elbow plot 后手动填入）；
# 默认依据=None，需看 sanity PCA 的 elbow plot 确定（通常 15-30）；
# 调大调小=太少丢失信号、太多引入噪声，elbow 处（曲线斜率显著变缓）是最优选择；
# 何时改=每次跑完 sanity PCA 后根据 elbow plot 决定，填入后再继续运行 checkpoint。
N_PCS_FINAL = None  # PI 需在看完 elbow plot 后手动填入


### 计算与开关（方法开关 / 计算资源）


In [ ]:
# === PARAMS ===
# === 组三：计算与开关（方法开关 / 计算资源）===

# HVG_SUBSAMPLE_PER_BATCH：是什么=每批先采样 N 细胞再选 HVG；
# 默认依据=None 全量；
# 调大调小=设值可缓解细胞数不平衡致大数据集主导 HVG，过小则不稳定（notebook 已带 3-seed Jaccard 稳定性检验）；
# 何时该改=各 batch 细胞数悬殊时设为 min(各 batch 细胞数) 或 500-2000。
HVG_SUBSAMPLE_PER_BATCH = None

# RANDOM_SEED：随机种子；固定它保证 HVG 子采样/稳定性检验可复现；改它只为做稳定性对照，不改科学结论。
RANDOM_SEED    = 42


### 输出与运行标识

这组是运行标识/产物版本，非科学判断，改 RUN_ID 即可创建新 run 而不覆盖旧结果。


In [ ]:
# === PARAMS ===
# === 组四：输出与运行标识 ===
# 每次调参改用新 RUN_ID，禁止覆盖旧 run（UX-3 保留策略）。
RUN_ID = "03-normalized-v1-run001"
RUN_ROOT = "results/runs"
OUTPUT_FILENAME = "03_normalized_v1.h5ad"
OUTPUT_VERSION = 1


## 回跑与迭代（调参机制）

### 在管线中的位置
- **上游**：02（跨数据集合并），读 `02_merged_v*.h5ad`
- **下游**：04（嵌入降维）、05（Leiden 聚类）、06（细胞类型注释）全部依赖本 stage 的标准化结果与 HVG 选择
- **关键地位**：03 是管线中第一个直接影响下游所有 stage 的节点——标准化方法与 HVG 数量一旦确定，04-06 的 PCA、UMAP、聚类、注释全部基于此

### 为什么要回跑？
标准化和 HVG 选择的参数看似"一次设定即可"，但实际问题往往在下游才暴露：

- **HVG 数量不足或过多**：如果在 06 注释时发现已知的细胞类型 marker 基因不在 HVG 列表中（例如 `MUC5AC` 未入选导致无法区分黏液细胞与主细胞），说明 `N_TOP_GENES` 偏小；如果 HVG 中充斥核糖体/线粒体基因（在 HVG 组成分析中可见），说明排除列表需要更新
- **标准化方法不当导致批次效应残留**：如果在 04 的 PCA 图中发现 PC1 与文库大小强相关（`total_counts`），或 05 聚类后各 batch（数据集来源）完全分离，可能是 `standard` 标准化对批次效应稳健性不足，需切换到 `pearson_residuals`
- **HVG 选择对参数过于敏感**：Sweep 模式下 Jaccard 相似度热力图显示不同 `N_TOP_GENES` 值间 HVG 重叠度低（<0.7），说明下游结果对 HVG 参数敏感，应慎重选择或扩大 HVG 集合
- **Elbow plot 无明确肘部**：PC 方差贡献曲线平缓下降、无明确拐点，提示数据信号弱或噪声高，可能需要回到 01 调整 QC 阈值或检查上游数据质量

### 如何回跑（三步操作）
1. **改 `UPSTREAM_RUN_ID`**——如需切换上游 02 版本（例如 02 合并了新数据集或有更新的合并策略）
2. **改 `RUN_ID`**——每次调参使用新的、不可覆盖的运行标识（例如 `03-normalized-v2-run001`）
3. **调整参数**——在下方 PARAMS cell 中修改 `N_TOP_GENES`、`NORMALIZATION_METHOD`、`HVG_FLAVOR` 等参数 → 重跑本 notebook（Kernel → Restart & Run All）

### 版本约定
- **`draft`**：本 notebook 计算出的候选标准化结果，固定为 `NEEDS_REVIEW`
- **`promoted`**：研究者审查并明确选择后，才可提升为下游权威输入
- **下游取数**：Stage 04 只能读取 promoted Stage 03；当前 draft 不可消费

### 追溯链
本 notebook 在写出前自动记录以下字段到 `adata.uns`：
- `stage` = `"03_normalized"`（本 stage 标识）
- `status` = `"NEEDS_REVIEW"`（禁止手工改为 `"SUCCESS"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_VERSION` 一致的版本号

如需查询"03 有哪些版本？哪些依赖 02_v1？"，可直接检查 `results/runs/<RUN_ID>/draft/manifest.json`。


In [ ]:
# === Setup：sys.path + 导入依赖 ===
# sys.path 必须在 scanpy 导入之前设置，否则找不到 scrna_integration 模块。
import sys, os, json
from pathlib import Path

_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

# --- 所有 import 集中在这里 ---
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import gc
import re
import itertools
from scrna_integration.run_contract import (
    atomic_write_json, collect_runtime_provenance, determine_stage_status, prepare_run,
    promote_run, resume_run, sha256_file, snapshot_effective_parameters, validate_checkpoint,
    validate_expression_contract,
)

np.random.seed(RANDOM_SEED)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

# 环境自检（每次运行自动检测平台 + conda 环境 + 关键包）
try:
    from scrna_integration.platform import env_check
    env_check()
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}）")

## Preflight：执行前集中校验（不载入 X）

在读大 h5ad、跑标准化之前，一次性校验输入存在、上游契约对得上、参数合法，把「能跑但结论不可信」和「跑到一半才报错」挡在最前面。任一不合法立即 raise 明确错误。


In [ ]:
# === Preflight：执行前集中校验（backed 模式，不载入 X）===
# 依赖 03-setup 已导入的 json / Path / sc / np 与 run_contract 各函数。

print("===== Preflight =====")

# (1) 参数类型/取值合法性（纯本地判断，最快失败，不读任何文件）
# NORMALIZATION_METHOD
_valid_methods = {"standard", "pearson_residuals"}
if NORMALIZATION_METHOD not in _valid_methods:
    raise ValueError(
        f"NORMALIZATION_METHOD='{NORMALIZATION_METHOD}' 不合法，"
        f"允许值：{sorted(_valid_methods)}"
    )

# TARGET_SUM
if not isinstance(TARGET_SUM, (int, float)) or TARGET_SUM <= 0:
    raise ValueError(f"TARGET_SUM={TARGET_SUM} 必须为正数（int/float，>0）")

# N_TOP_GENES：单值 int>0 或非空 list[int>0]
if isinstance(N_TOP_GENES, list):
    if len(N_TOP_GENES) == 0:
        raise ValueError("N_TOP_GENES 为列表时不可为空")
    for v in N_TOP_GENES:
        if not isinstance(v, int) or v <= 0:
            raise ValueError(f"N_TOP_GENES 列表元素必须为正 int，非法值：{v}")
elif not isinstance(N_TOP_GENES, int) or N_TOP_GENES <= 0:
    raise ValueError(f"N_TOP_GENES={N_TOP_GENES} 必须为正 int 或非空 list[int>0]")

# HVG_FLAVOR：str 或非空 list[str]，且每个值在合法集合内
_valid_flavors = {"seurat", "seurat_v3", "cell_ranger"}
if isinstance(HVG_FLAVOR, list):
    if len(HVG_FLAVOR) == 0:
        raise ValueError("HVG_FLAVOR 为列表时不可为空")
    for v in HVG_FLAVOR:
        if v not in _valid_flavors:
            raise ValueError(
                f"HVG_FLAVOR 中 '{v}' 不合法，允许值：{sorted(_valid_flavors)}"
            )
elif not isinstance(HVG_FLAVOR, str):
    raise ValueError(f"HVG_FLAVOR={HVG_FLAVOR!r} 必须为 str 或 list[str]")
elif HVG_FLAVOR not in _valid_flavors:
    raise ValueError(
        f"HVG_FLAVOR='{HVG_FLAVOR}' 不合法，允许值：{sorted(_valid_flavors)}"
    )

# bool 类型检查
_bool_params = {
    "BATCH_AWARE_HVG": BATCH_AWARE_HVG,
    "EXCLUDE_MT_FROM_HVG": EXCLUDE_MT_FROM_HVG,
    "EXCLUDE_RIBO_FROM_HVG": EXCLUDE_RIBO_FROM_HVG,
    "EXCLUDE_HB_FROM_HVG": EXCLUDE_HB_FROM_HVG,
    "EXCLUDE_CELL_CYCLE_FROM_HVG": EXCLUDE_CELL_CYCLE_FROM_HVG,
    "SCALE": SCALE,
}
for name, val in _bool_params.items():
    if not isinstance(val, bool):
        raise ValueError(f"{name}={val!r} 必须为 bool，实际为 {type(val).__name__}")

# HVG_BATCH_KEY：非空 str（真正是否在 obs 里由 03-hvg-sweep 校验，preflight 不载数据不重复）
if not isinstance(HVG_BATCH_KEY, str) or not HVG_BATCH_KEY.strip():
    raise ValueError(f"HVG_BATCH_KEY={HVG_BATCH_KEY!r} 必须为非空 str")

# list 类型检查
_list_params = {
    "CUSTOM_EXCLUDE_PATTERNS": CUSTOM_EXCLUDE_PATTERNS,
    "FORCED_INCLUDE_GENES": FORCED_INCLUDE_GENES,
    "REGRESS_OUT": REGRESS_OUT,
}
for name, val in _list_params.items():
    if not isinstance(val, list):
        raise ValueError(f"{name}={val!r} 必须为 list，实际为 {type(val).__name__}")

# HVG_SUBSAMPLE_PER_BATCH：None 或正 int
if HVG_SUBSAMPLE_PER_BATCH is not None:
    if not isinstance(HVG_SUBSAMPLE_PER_BATCH, int) or HVG_SUBSAMPLE_PER_BATCH <= 0:
        raise ValueError(f"HVG_SUBSAMPLE_PER_BATCH={HVG_SUBSAMPLE_PER_BATCH} 必须为 None 或正 int")

# MAX_SCALE_VALUE：正数
if not isinstance(MAX_SCALE_VALUE, (int, float)) or MAX_SCALE_VALUE <= 0:
    raise ValueError(f"MAX_SCALE_VALUE={MAX_SCALE_VALUE} 必须为正数")

# OUTPUT_VERSION：正 int
if not isinstance(OUTPUT_VERSION, int) or OUTPUT_VERSION <= 0:
    raise ValueError(f"OUTPUT_VERSION={OUTPUT_VERSION} 必须为正 int")

# RANDOM_SEED：int
if not isinstance(RANDOM_SEED, int):
    raise ValueError(f"RANDOM_SEED={RANDOM_SEED!r} 必须为 int，实际为 {type(RANDOM_SEED).__name__}")

# pearson_residuals + SCALE=True 组合 warning（不阻断，仅提示）
if NORMALIZATION_METHOD == "pearson_residuals" and SCALE:
    print("\u26a0  Pearson residuals 后再 scale 通常无意义——残差已方差稳定化，建议 SCALE=False")

print("(1) 参数类型/取值 \u2713")

# (2) 上游输入存在性（只读 manifest.json，不读 h5ad）
_manifest_path = Path(UPSTREAM_RUN_ROOT) / UPSTREAM_RUN_ID / "promoted" / "manifest.json"
if not _manifest_path.exists():
    raise FileNotFoundError(
        f"上游 manifest 不存在：{_manifest_path}\n"
        f"请确认已执行 02_merged notebook 并 promote checkpoints。"
    )

_manifest = json.loads(_manifest_path.read_text(encoding="utf-8"))
if _manifest.get("run_id") != UPSTREAM_RUN_ID:
    raise ValueError(
        f"manifest.run_id='{_manifest.get('run_id')}' 与配置 UPSTREAM_RUN_ID='{UPSTREAM_RUN_ID}' 不匹配"
    )
if _manifest.get("stage") != "02_merged":
    raise ValueError(
        f"manifest.stage='{_manifest.get('stage')}'，期望 '02_merged'。"
        f"请确认上游是 02 合并阶段产物。"
    )

# 从 manifest 取 checkpoint 路径并校验文件存在
_checkpoint_path = None
# promoted 目录下的 checkpoint 产物
_promoted_dir = Path(UPSTREAM_RUN_ROOT) / UPSTREAM_RUN_ID / "promoted"
_checkpoint_candidates = sorted(_promoted_dir.glob("*.h5ad"))
if _checkpoint_candidates:
    _checkpoint_path = _checkpoint_candidates[0]
elif "checkpoint" in _manifest and isinstance(_manifest["checkpoint"], dict):
    _cp_rel = _manifest["checkpoint"].get("path", "")
    _cp_full = _promoted_dir / _cp_rel
    if _cp_full.exists():
        _checkpoint_path = _cp_full

if _checkpoint_path is None:
    raise FileNotFoundError(
        f"未在 {_promoted_dir} 找到上游 .h5ad checkpoint 文件。"
        f"请确认 02_merged 已完成并 promote。"
    )

print(f"(2) 上游 manifest + checkpoint 存在 \u2713  ({_checkpoint_path.name})")

# (3) 上游 expression_contract stage==02（backed 模式，只读元数据，X 不进内存）
# backed="r" 只读 metadata（uns/obs/var）+ layers 键名，不把 X 载入内存
_handle = sc.read_h5ad(str(_checkpoint_path), backed="r")
try:
    _upstream_contract = _handle.uns.get("expression_contract")
    if _upstream_contract is None:
        raise KeyError(
            "上游 h5ad 缺少 expression_contract，无法校验。"
            "请确认上游 notebook 已执行 P0 counts 契约修复。"
        )
    # 校验上游契约与当前预期一致
    validate_expression_contract(
        _handle, expected_scale="raw_counts", stage="02"
    )
    # 校验 layers["counts"] 键存在（backed 下只查键名，不取数据）
    if "counts" not in _handle.layers:
        raise KeyError(
            "上游 h5ad 缺少 layers['counts']——未建立原始计数层。"
            "请确认上游 notebook 已执行 P0 counts 契约修复（决策 2）。"
        )
finally:
    _handle.file.close()

print("(3) expression_contract stage==02 + layers['counts'] \u2713")

# SoupX / doublet 开关校验在 03 为 N/A（那是 01 的职责），不做空校验。

# (4) 生效参数回显：让研究者执行前肉眼确认关键参数
print("\n----- 生效参数 -----")
print(f"  NORMALIZATION_METHOD = {NORMALIZATION_METHOD}")
print(f"  TARGET_SUM           = {TARGET_SUM}")
print(f"  N_TOP_GENES          = {N_TOP_GENES}")
print(f"  HVG_FLAVOR           = {HVG_FLAVOR}")
print(f"  BATCH_AWARE_HVG      = {BATCH_AWARE_HVG}")
print(f"  HVG_BATCH_KEY        = {HVG_BATCH_KEY}")
print(f"  EXCLUDE_MT/RIBO/HB   = {EXCLUDE_MT_FROM_HVG}/{EXCLUDE_RIBO_FROM_HVG}/{EXCLUDE_HB_FROM_HVG}")
print(f"  EXCLUDE_CELL_CYCLE   = {EXCLUDE_CELL_CYCLE_FROM_HVG}")
print(f"  SCALE                = {SCALE}")
print(f"  RUN_ID               = {RUN_ID}")
print(f"  OUTPUT_VERSION       = {OUTPUT_VERSION}")
print("----- 生效参数完 -----\n")

print("\u2713 Preflight 通过")

In [ ]:
# === 加载上游 ===
upstream_run = resume_run(UPSTREAM_RUN_ROOT, UPSTREAM_RUN_ID, promoted=True)
upstream_manifest_path = upstream_run.promoted_dir / "manifest.json"
upstream_manifest = json.loads(upstream_manifest_path.read_text(encoding="utf-8"))
if upstream_manifest.get("run_id") != UPSTREAM_RUN_ID:
    raise ValueError("upstream manifest run_id 与配置不匹配")
if upstream_manifest.get("stage") != "02_merged":
    raise ValueError("upstream manifest stage 必须是 02_merged")
UPSTREAM_CHECKPOINT = validate_checkpoint(upstream_manifest_path)
upstream_input = {
    "run_id": UPSTREAM_RUN_ID, "stage": "02_merged",
    "manifest_path": str(upstream_manifest_path),
    "manifest_sha256": sha256_file(upstream_manifest_path),
    "checkpoint_path": str(UPSTREAM_CHECKPOINT),
    "checkpoint_sha256": sha256_file(UPSTREAM_CHECKPOINT),
}
print("Loading verified upstream:", UPSTREAM_CHECKPOINT)
adata = sc.read_h5ad(UPSTREAM_CHECKPOINT)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# F2修复：全量整数检查——从「采样+warning」升级为「全矩阵校验+阻断」
# 此前只采样 X[:500,:500] 且仅 warning 不阻断，拦不住意外二次归一
# 对大规模 sparse 矩阵用 row-wise 抽样校验（check_rows 行），避免 OOM
# 修复：整数检查必须在检查 preprocessing_done 之后——若上游已完成标准化则跳过，
# 否则 CELL 7 的 skip 逻辑不会被触达
_pp_done_check = adata.uns.get("preprocessing_done", [])
if "normalization" in _pp_done_check:
    print("上游 preprocessing_done 含 normalization，跳过整数检查（上游已完成归一化，非整数数据属正常）")
else:
    check_rows = 2000
    _total_rows = adata.n_obs
    rng = np.random.default_rng(RANDOM_SEED)
    _sample_indices = rng.choice(_total_rows, size=min(check_rows, _total_rows), replace=False)
    _sample_indices.sort()
    _sample_x = adata.X[_sample_indices, :]
    if sp.issparse(_sample_x):
        _sample_x = _sample_x.toarray()
    _is_integer_values = np.allclose(_sample_x, np.round(_sample_x), atol=0.01)

    if _is_integer_values:
        print(f"X 含整数值 counts（dtype={adata.X.dtype}，抽样 {len(_sample_indices)} 行校验通过），符合上游合约")
    else:
        # 全量矩阵非整数 —— 硬阻断，防止二次归一
        raise ValueError(
            f"03_normalized 检测到 X 矩阵含非整数值（dtype={adata.X.dtype}），"
            f"上游可能已做过标准化变换。如需跳过标准化，请确认 02_merged 传递了"
            f"adata.uns['preprocessing_done'] 且其中含 'normalization'。"
            f"禁止对已归一数据二次归一（会影响生物学结论可信度）。"
        )


## 验证原始计数层

下游方法（scVI、scANVI、DESeq2、pseudobulk）需要原始整数计数来精准建模。
依照 counts 契约（决策 1/2），`layers["counts"]` 由上游 01 建立、02 拼接传播，
是框架内唯一的权威原始计数位置。03 只读取验证，不覆盖也不自行复制。

In [ ]:
# 从上游验证 layers["counts"]（权威原始计数层，由 01 建立、02 传播）。
# 03 不再从 adata.X 自行拷贝——X 在上游可能已被变换，
# 只有 layers["counts"] 是可依赖的原始整数计数。
# 上游契约签名在 adata.uns["expression_contract"] 中。

# 验证 expression_contract 存在且字段合法
_upstream_contract = validate_expression_contract(adata, expected_scale="raw_counts", stage="02")

if "counts" not in adata.layers:
    raise KeyError(
        "layers['counts'] 不存在——上游 01/02 未建立原始计数层。"
        "请确认上游 notebook 已执行 P0 counts 契约修复（决策 2）。"
    )

_counts_layer = adata.layers["counts"]
print(f"layers['counts'] 已从上个 stage 传播: shape={_counts_layer.shape}, "
      f"dtype={_counts_layer.dtype}, sparse={sp.issparse(_counts_layer)}")

# 验证 layers["counts"] 与 expression_contract 的 counts_layer 字段一致
_contract_counts_layer = _upstream_contract.get("counts_layer")
if _contract_counts_layer != "counts":
    raise ValueError(
        f"expression_contract.counts_layer 期望 'counts'，实际为 {_contract_counts_layer!r}"
    )
print(f"expression_contract.counts_layer='{_contract_counts_layer}' ✓")
print(f"counts_source={_upstream_contract.get('counts_source')!r}, "
      f"counts_validated={_upstream_contract.get('counts_validated')}, "
      f"counts_integer_check={_upstream_contract.get('counts_integer_check')!r}")

## 标准化

两种方法路线可选（通过 `NORMALIZATION_METHOD` 切换）：

- **`standard`**（默认）：`normalize_total` 将每个细胞缩放至相同总 UMI，消除测序深度差异；
  接着 `log1p` 做方差稳定化，将右偏分布拉近正态，使高表达基因不过度主导 PCA。
  这是单细胞领域的经典路线，适用绝大多数场景。
- **`pearson_residuals`**（Lause et al. 2021, Genome Biology）：基于负二项模型的残差变换，
  一步完成方差稳定化，不需要显式 log1p。优点是对 dropout 更鲁棒；
  缺点是不保留 log-normalized 空间的可解释性，且函数在 `sc.experimental` 中。


### Pearson Residuals 注意事项

如果选择 `pearson_residuals`：
- 这是实验性方法（`sc.experimental.pp`），scanpy API 可能在未来版本变动
- 不适合极小数据集（<500 cells），需要足够样本来稳定估计负二项分布参数
- 输出是 residuals（非 log 空间），下游工具需兼容（scVI/Harmony 兼容，部分工具可能不兼容）


In [ ]:
# === 标准化分支 ===
# P0 counts 契约（决策 2）：layers["counts"] 一经 01 建立不得被 normalization 覆盖。
# 标准化前记录 float64 sum，标准化后断言相对差 < 1e-6。

# 标准化前：计算 layers["counts"] 的 float64 总和的基准
_counts_sum_before = float(adata.layers["counts"].sum(dtype=np.float64))

# 检查上游契约状态，决定是否需要标准化
_skip_normalize = False
_upstream_x_scale = _upstream_contract.get("x_scale", "raw_counts")
if _upstream_x_scale != "raw_counts":
    _skip_normalize = True
    print(f"上游 expression_contract.x_scale={_upstream_x_scale!r}，非 raw_counts，跳过标准化")

if not _skip_normalize:
    if NORMALIZATION_METHOD == "standard":
        # normalize_total：每个细胞缩放至相同总 UMI，消除测序深度差异
        sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
        # log1p：log(1+x) 方差稳定化，使高表达基因不过度主导 PCA
        sc.pp.log1p(adata)
        print(f"✓ 标准化完成：normalize_total(target_sum={TARGET_SUM}) + log1p")
        print(f"  标准化后 X mean={adata.X.mean():.4f}, max={adata.X.max():.4f}")
    elif NORMALIZATION_METHOD == "pearson_residuals":
        # Pearson residuals（Lause et al. 2021, Genome Biology）
        # 一步完成方差稳定化，不需要单独 log1p
        # 优点：对 dropout 更鲁棒；
        # 注意：sc.experimental API 可能在 scanpy 未来版本中变动
        sc.experimental.pp.normalize_pearson_residuals(adata)
        print("✓ 标准化完成：Pearson residuals (Lause 2021)")
        print(f"  标准化后 X mean={adata.X.mean():.4f}, std={adata.X.std():.4f}")
    else:
        raise ValueError(f"不支持的 NORMALIZATION_METHOD: {NORMALIZATION_METHOD}，请使用 'standard' 或 'pearson_residuals'")

    # 转为 float32——内存减半，单细胞数据有效精度无实质影响
    adata.X = adata.X.astype(np.float32)
    print(f"X dtype 已转为: {adata.X.dtype}")
else:
    print(f"标准化已跳过：上游 expression_contract.x_scale={_upstream_x_scale!r}")
    # 跳过 normalize_total + log1p，但 float32 转换仍然需要（上游可能是 float64）
    adata.X = adata.X.astype(np.float32)
    print(f"X dtype 已转为: {adata.X.dtype}（跳过标准化，仅做类型转换）")

# 标准化后：验证 layers["counts"] 未被修改（契约核心约束）
_counts_sum_after = float(adata.layers["counts"].sum(dtype=np.float64))
_rel_diff = abs(_counts_sum_before - _counts_sum_after) / max(abs(_counts_sum_before), 1.0)
assert _rel_diff < 1e-6, (
    f"layers['counts'] 在标准化过程中被修改！"
    f" sum before={_counts_sum_before:.1f}, after={_counts_sum_after:.1f}, "
    f"rel_diff={_rel_diff:.2e}"
)
_counts_integrity_checked = True
print(f"layers['counts'] 完整性验证通过：sum before={_counts_sum_before:.1f}, "
      f"after={_counts_sum_after:.1f}, rel_diff={_rel_diff:.2e}")

# === 归一化决策诊断（platform 决策可见原则）===
# 无论跳过还是执行，均在 cell 输出区打印决策结果
print("\n===== 归一化决策 =====")
if _skip_normalize:
    print("跳过归一化（上游已完成）")
elif NORMALIZATION_METHOD == "standard":
    print("已执行归一化（sc.pp.normalize_total + log1p）")
elif NORMALIZATION_METHOD == "pearson_residuals":
    print("已执行归一化（Pearson residuals, Lause 2021）")
else:
    print(f"已执行归一化（{NORMALIZATION_METHOD}）")

## 标准化效果可视化

**看什么**：左图展示标准化前各细胞的总 UMI 计数分布（来自 counts layer）——
不同细胞测序深度差异悬殊。右图展示标准化后的表达值分布——
standard 模式下所有细胞缩放到统一尺度，pearson_residuals 模式下残差围绕 0 对称分布。

In [ ]:
# === 标准化前后对比图 ===
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：标准化前——从 counts layer 看每个细胞的总 UMI 分布
counts_before = np.array(adata.layers["counts"].sum(axis=1)).flatten()
axes[0].hist(counts_before, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(np.median(counts_before), color="red", linestyle="--",
                label=f'中位={np.median(counts_before):.0f}')
axes[0].set_xlabel("总 UMI 计数")
axes[0].set_ylabel("细胞数")
axes[0].set_title("标准化前\n各细胞总 UMI 计数差异大")
axes[0].legend(fontsize=9)

# 右图：标准化后——表达值分布
if sp.issparse(adata.X):
    sample_vals = adata.X.data[:200000] if len(adata.X.data) > 200000 else adata.X.data
else:
    # Dense 矩阵（Pearson residuals 可能产出 dense）——随机采样避免 OOM
    # .flatten() 对 100k×30k=3B float32≈12GB 直接 OOM；随机采 20 万点足够画直方图
    _flat_size = adata.X.shape[0] * adata.X.shape[1]
    if _flat_size > 200000:
        _sample_idx = np.random.choice(_flat_size, size=200000, replace=False)
        sample_vals = adata.X.ravel()[_sample_idx]
    else:
        sample_vals = adata.X.ravel()
axes[1].hist(sample_vals, bins=100, color="coral", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("标准化后表达值")
axes[1].set_ylabel("频数")
axes[1].set_title(f"标准化后（{NORMALIZATION_METHOD}）\n表达值分布")
axes[1].axvline(0, color="gray", linestyle=":", alpha=0.5)

plt.suptitle("标准化效果", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/03_normalize_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

# --- standard 模式额外诊断：top-3 基因标准化前后分布对比 ---
# 这是经典的 log1p 效果展示——看 log1p 如何将右偏分布拉近正态
if NORMALIZATION_METHOD == "standard":
    # 基于原始 counts 均值排名前 3 的基因
    top_idx = np.argsort(
        np.array(adata.layers["counts"].mean(axis=0)).flatten()
    )[-3:]
    top_genes = adata.var_names[top_idx].tolist()
    print(f"\ntop-3 基因表达分布对比: {top_genes}")

    # 重新计算 pre-log1p 标准化值（normalize_total 后、log1p 前）
    raw_top = adata.layers["counts"][:, top_idx].toarray()
    lib_size = np.array(adata.layers["counts"].sum(axis=1)).flatten()
    norm_expr = raw_top / lib_size[:, None] * TARGET_SUM

    # post-log1p 直接读取当前 adata.X
    log_expr = adata.X[:, top_idx].toarray()

    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
    for i, gene in enumerate(top_genes):
        axes2[0].hist(norm_expr[:, i], bins=50, alpha=0.5, label=gene, density=True)
        axes2[1].hist(log_expr[:, i], bins=50, alpha=0.5, label=gene, density=True)
    axes2[0].set_xlabel("标准化后表达量（未 log）")
    axes2[0].set_ylabel("密度")
    axes2[0].set_title("Log1p 前：高度右偏\n少数细胞极高值主导")
    axes2[0].legend(fontsize=8)
    axes2[1].set_xlabel("log1p 表达量")
    axes2[1].set_ylabel("密度")
    axes2[1].set_title("Log1p 后：接近正态\n适合 PCA 等线性方法")
    axes2[1].legend(fontsize=8)
    plt.suptitle("Log1p 方差稳定化效果", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/figures/03_log1p_before_after.png", dpi=150, bbox_inches="tight")
    plt.show()

## 高可变基因选择（Scalar-or-Sweep 双模）

大多数基因在所有细胞中表达量相近（管家基因），不携带区分细胞类型的信息。
高可变基因（HVG）是那些在不同细胞间表达差异最大的基因——
它们携带了细胞类型、状态差异的核心信号。

**双模设计**：
- **单值模式**：`N_TOP_GENES=2000`，直接选一组 HVG，干净利落
- **Sweep 模式**：`N_TOP_GENES=[1500, 2000, 3000]`，自动遍历所有参数组合，
  输出 Jaccard 重叠度热力图，帮助 PI 判断参数敏感性

**批次感知 HVG**：当 `BATCH_AWARE_HVG=True` 时，scanpy 在每个 batch 内独立选 HVG，
取跨批次并集并按出现频率排名。这防止某个数据集的技术噪声基因因全局高变而被选中——
是多数据集整合的关键 trick。

In [ ]:
# === HVG 选择：Scalar-or-Sweep 双模 ===
# 写单值直接跑，写列表自动对比不同参数组合的 HVG 集合

_n_genes_values = N_TOP_GENES if isinstance(N_TOP_GENES, list) else [N_TOP_GENES]
_flavor_values = HVG_FLAVOR if isinstance(HVG_FLAVOR, list) else [HVG_FLAVOR]

# --- Pearson residuals 兼容性 ---
# seurat flavor 假设 count 数据的 mean-variance 关系，
# 对 Pearson 残差（均值~0、可为负值）不兼容。自动切换到 seurat_v3（基于方差排名）
if NORMALIZATION_METHOD == "pearson_residuals":
    _flavor_values_safe = []
    for f in _flavor_values:
        if f == "seurat":
            print("⚠️ Pearson residuals + seurat flavor 不兼容（seurat 假设 count 数据），自动切换到 seurat_v3")
            _flavor_values_safe.append("seurat_v3")
        else:
            _flavor_values_safe.append(f)
    _flavor_values = _flavor_values_safe

hvg_results = {}  # 存储各组合的结果供对比

# HVG 等量子采样：平衡各 batch 对 HVG 选择的影响
# 当数据集细胞数严重不平衡时，大 batch 会主导 HVG 排名——采样子集后再选可缓解
if HVG_SUBSAMPLE_PER_BATCH is not None and BATCH_AWARE_HVG:
    np.random.seed(RANDOM_SEED)
    batch_col = adata.obs[HVG_BATCH_KEY]
    subsample_idx = []
    for batch in batch_col.unique():
        batch_idx = adata.obs_names[batch_col == batch]
        n_sample = min(HVG_SUBSAMPLE_PER_BATCH, len(batch_idx))
        subsample_idx.extend(np.random.choice(batch_idx, size=n_sample, replace=False))
    adata_hvg = adata[subsample_idx].copy()
    print(f"HVG 子采样：每 batch 取 {HVG_SUBSAMPLE_PER_BATCH} 细胞（实际 {len(subsample_idx)} 总细胞）用于 HVG 选择")
else:
    adata_hvg = adata

print(f"HVG Sweep: n_genes={_n_genes_values}, flavor={_flavor_values}")
print(f"BATCH_AWARE_HVG={BATCH_AWARE_HVG}", end="")
if BATCH_AWARE_HVG:
    print(f", batch_key={HVG_BATCH_KEY}")
else:
    print()
print()

for n_genes in _n_genes_values:
    for flavor in _flavor_values:
        label = f"n={n_genes}, flavor={flavor}"
        print(f"--- {label} ---")

        # batch-aware HVG：每个 batch 独立选 HVG，取并集按跨 batch 出现频率排名
        if BATCH_AWARE_HVG:
            if HVG_BATCH_KEY not in adata.obs.columns:
                raise KeyError(
                    f"HVG_BATCH_KEY='{HVG_BATCH_KEY}' 不在 obs 列中。"
                    f"可用列: {list(adata.obs.columns)[:10]}..."
                    f"请检查 BATCH_AWARE_HVG 和 HVG_BATCH_KEY 参数。"
                )
            import warnings

            _n_batches = adata.obs[HVG_BATCH_KEY].nunique()
            if _n_batches < 2:
                warnings.warn(
                    f"检测到 HVG_BATCH_KEY='{HVG_BATCH_KEY}' 仅包含 {_n_batches} 个批次"
                    f"（唯一值 = {_n_batches}），但 BATCH_AWARE_HVG = True。"
                    f"batch-aware HVG 在每个批次内独立选取高可变基因后取并集——"
                    f"单批次下等同于普通 HVG，batch_aware 标注将产生误导。"
                    f"建议：将 BATCH_AWARE_HVG 设为 False 以明确意图，"
                    f"或确认合并了多个数据集后再启用。"
                    f"当前管线将以普通 HVG 模式继续运行。",
                    UserWarning,
                )
            if flavor == "seurat_v3":
                # seurat_v3 内部使用负二项模型，要求原始 counts（非归一化数据）
                # 传入 layer="counts" 确保从权威原始计数层读取，而非归一化后的 X
                sc.pp.highly_variable_genes(
                    adata_hvg, n_top_genes=n_genes, flavor="seurat_v3",
                    layer="counts", batch_key=HVG_BATCH_KEY
                )
            else:
                # seurat / cell_ranger 均在归一化后的 X 上计算 mean-variance
                sc.pp.highly_variable_genes(
                    adata_hvg, n_top_genes=n_genes, flavor=flavor,
                    batch_key=HVG_BATCH_KEY
                )
        else:
            if flavor == "seurat_v3":
                # seurat_v3 要求原始 counts，从 layer 读取
                sc.pp.highly_variable_genes(
                    adata_hvg, n_top_genes=n_genes, flavor="seurat_v3",
                    layer="counts"
                )
            elif flavor in ("seurat", "cell_ranger"):
                # seurat / cell_ranger 用归一化后的 X
                sc.pp.highly_variable_genes(
                    adata_hvg, n_top_genes=n_genes, flavor=flavor
                )
            else:
                raise ValueError(f"不支持的 HVG_FLAVOR: {flavor!r}，允许值: seurat / seurat_v3 / cell_ranger")

        # 子采样模式下将子集 HVG 投影回全数据
        if adata_hvg is not adata:
            adata.var["highly_variable"] = adata.var_names.isin(
                adata_hvg.var_names[adata_hvg.var["highly_variable"]]
            )

        # 存储结果到带参数后缀的 var 列，供后续对比
        key = f"hvg_{flavor}_{n_genes}"
        adata.var[key] = adata.var["highly_variable"].copy()
        n_selected = int(adata.var[key].sum())
        hvg_results[key] = {"n_genes": n_genes, "flavor": flavor, "n_selected": n_selected}
        print(f"  {key}: {n_selected} genes selected")

print(f"\n共生成 {len(hvg_results)} 组 HVG 结果")
print(f"adata.var['highly_variable'] 当前指向最后组合: {list(hvg_results.keys())[-1]}")

# 清理子采样临时对象（释放 adata 子集占用的内存）
if adata_hvg is not adata:
    del adata_hvg

# --- 子采样稳定性验证 ---
# 3 次不同 seed，检查 HVG 重叠度。如果 Jaccard < 0.8 → 建议增大 HVG_SUBSAMPLE_PER_BATCH
if HVG_SUBSAMPLE_PER_BATCH is not None:
    _stability_sets = []
    for _seed in [RANDOM_SEED, RANDOM_SEED + 1, RANDOM_SEED + 2]:
        np.random.seed(_seed)
        _sub_idx = []
        for batch in adata.obs[HVG_BATCH_KEY].unique():
            batch_idx = adata.obs_names[adata.obs[HVG_BATCH_KEY] == batch]
            n_s = min(HVG_SUBSAMPLE_PER_BATCH, len(batch_idx))
            _sub_idx.extend(np.random.choice(batch_idx, size=n_s, replace=False))
        _tmp = adata[_sub_idx].copy()
        sc.pp.highly_variable_genes(_tmp, n_top_genes=_n_genes_values[-1], flavor=_flavor_values[-1],
                                    batch_key=HVG_BATCH_KEY)
        _stability_sets.append(set(_tmp.var_names[_tmp.var["highly_variable"]]))
        del _tmp
    jaccards = []
    for s1, s2 in itertools.combinations(_stability_sets, 2):
        jaccards.append(len(s1 & s2) / len(s1 | s2))
    mean_j = np.mean(jaccards)
    print(f"HVG 子采样稳定性：3 次 Jaccard = {[f'{j:.3f}' for j in jaccards]} (mean={mean_j:.3f})")
    if mean_j < 0.8:
        print(f"  ⚠️ 子采样不稳定（Jaccard < 0.8），建议增大 HVG_SUBSAMPLE_PER_BATCH")
    else:
        print(f"  ✓ 子采样稳定（Jaccard ≥ 0.8）")

# === Sweep 模式：确认最终 HVG 选择 ===
# sweep 模式默认沿用循环最后一组参数的 highly_variable；
# 打印当前使用的 key，PI 可按需切换
if len(hvg_results) > 1:
    FINAL_HVG_KEY = list(hvg_results.keys())[-1]
    print(f"\nSweep 模式：当前 highly_variable 指向 {FINAL_HVG_KEY}")
    print(f"如需切换，运行：adata.var['highly_variable'] = adata.var['<目标key>'].copy()")
    print(f"可用 key：{list(hvg_results.keys())}")
else:
    print("单组模式：HVG 已在 highly_variable 列")


## HVG Sweep 对比

当使用列表模式（多个 `N_TOP_GENES` 或 `HVG_FLAVOR` 值）时，
下方 cell 输出不同参数组合间 HVG 集合的 Jaccard 相似度热力图。

**解读**：
- 对角线 = 1.0（自身完全一致）
- Jaccard 接近 1.0 的 pair → 参数选择对该对不敏感，可放心任选
- Jaccard 显著低于 1.0 的 pair → HVG 集合对参数敏感，建议 PI 目视下游 UMAP 效果后决定

单值模式下此 cell 仅打印确认信息，不画图。

In [ ]:
# === HVG Sweep 对比：Jaccard 重叠度热力图 ===
if len(hvg_results) > 1:
    # Jaccard 相似度矩阵：不同参数组合间 HVG 集合的重叠度
    keys = list(hvg_results.keys())
    jaccard_matrix = pd.DataFrame(index=keys, columns=keys, dtype=float)

    for k1, k2 in itertools.combinations(keys, 2):
        set1 = set(adata.var_names[adata.var[k1]])
        set2 = set(adata.var_names[adata.var[k2]])
        j = len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0.0
        jaccard_matrix.loc[k1, k2] = j
        jaccard_matrix.loc[k2, k1] = j
    np.fill_diagonal(jaccard_matrix.values, 1.0)

    # 热力图
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(jaccard_matrix.values.astype(float), cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(keys, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(keys)))
    ax.set_yticklabels(keys, fontsize=9)
    plt.colorbar(im, ax=ax, label="Jaccard similarity")
    ax.set_title("HVG 集合重叠度（不同参数组合间）", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/figures/03_hvg_jaccard_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 汇总表
    print("\nHVG Sweep 汇总:")
    summary_df = pd.DataFrame(hvg_results).T
    from IPython.display import display as ipy_display
    ipy_display(summary_df)
else:
    entry = list(hvg_results.values())[0]
    print(f"✓ 单值模式：n_genes={entry['n_genes']}, flavor={entry['flavor']}, "
          f"选定 {entry['n_selected']} HVG")

## HVG 排除列表

某些基因类别表达量在不同细胞间确实高度可变，但这种可变性反映的是
**通用细胞状态**（如代谢活性、应激水平）或**技术背景**而非**细胞身份**——
它们不应驱动聚类和细胞类型鉴定。排除后依赖真正的细胞身份基因来区分细胞类型。

**排除类别**：
- **线粒体基因（MT-*）**：反映细胞膜破损/凋亡程度，是 QC 指标而非身份信号
- **核糖体基因（RPS*/RPL*）**：反映蛋白质合成活性，与细胞类型关联弱、批次间波动大
- **血红蛋白基因（HBA*/HBB*）**：红细胞污染标志，在组织样本中来自血液残留
- **自定义 pattern**：如免疫球蛋白（IG[HKL]）和 TCR（TR[ABGD]）基因——
  在淋巴细胞中高度可变，但反映的是克隆型而非细胞类型

**注意**：排除后**不做补位**。补位会把刚排除的基因类别中排名较低的成员重新拉回来，
削弱排除效果。如需精确某数量的 HVG，适当增大 `N_TOP_GENES` 初始值即可。

## 细胞周期评分（始终执行）

细胞周期评分（`sc.tl.score_genes_cell_cycle`）始终执行并将
`S_score`、`G2M_score`、`phase` 存入 `adata.obs`，无论
`EXCLUDE_CELL_CYCLE_FROM_HVG` 的值如何。

**为什么始终做评分？**
stage4（scVI 嵌入）可以将 `S_score`/`G2M_score` 作为 continuous
covariate 来消除周期效应，而无需从 HVG 中粗暴排除细胞周期基因。
这对研究增殖异常的项目（如肿瘤、炎-癌转化）尤为重要——
增殖信号本身就是重要的生物学信息。

**从 HVG 排除细胞周期基因**（`EXCLUDE_CELL_CYCLE_FROM_HVG=True`）
仅在增殖信号过度主导 PCA、掩盖细胞类型差异时作为最后手段启用。
默认保持 `False`。


In [ ]:
# === HVG 排除列表：构建排除 mask，移除后不补位 ===
exclude_mask = pd.Series(False, index=adata.var_names)
excluded_counts = {}

if EXCLUDE_MT_FROM_HVG:
    mt_mask = adata.var_names.str.upper().str.startswith("MT-")
    excluded_counts["MT"] = int((adata.var["highly_variable"] & mt_mask).sum())
    exclude_mask |= mt_mask

if EXCLUDE_RIBO_FROM_HVG:
    ribo_mask = adata.var_names.str.upper().str.match("^(RPS|RPL)")
    excluded_counts["Ribo"] = int((adata.var["highly_variable"] & ribo_mask).sum())
    exclude_mask |= ribo_mask

if EXCLUDE_HB_FROM_HVG:
    # 精确白名单：常见血红蛋白基因，避免 regex 误伤 HBEGF 等非血红蛋白基因
    HB_GENES_WHITELIST = ["HBA1", "HBA2", "HBB", "HBD", "HBE1", "HBG1", "HBG2", "HBM", "HBQ1", "HBZ"]
    hb_mask = adata.var_names.isin(HB_GENES_WHITELIST)
    excluded_counts["HB"] = int((adata.var["highly_variable"] & hb_mask).sum())
    exclude_mask |= hb_mask

for pattern in CUSTOM_EXCLUDE_PATTERNS:
    try:
        custom_mask = adata.var_names.str.contains(pattern, case=False, regex=True)
    except re.error as e:
        print(f"⚠ CUSTOM_EXCLUDE_PATTERNS 中 '{pattern}' 正则无效 ({e})，跳过")
        continue
    excluded_counts[pattern] = int((adata.var["highly_variable"] & custom_mask).sum())
    exclude_mask |= custom_mask

# === 细胞周期评分（始终执行，结果存 obs 供 stage4 作为 covariate 使用）===
# Tirosh et al. 2015 的 S 期和 G2M 期 marker genes
s_genes_cc = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
g2m_genes_cc = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]

# 评分需要 log-normalized 数据（standard 模式下当前 adata.X 已经是）
if NORMALIZATION_METHOD == "pearson_residuals":
    # score_genes_cell_cycle 需要 log-normalized 数据
    _tmp_cc = adata.copy()
    _tmp_cc.X = _tmp_cc.layers["counts"].copy()
    sc.pp.normalize_total(_tmp_cc, target_sum=1e4)
    sc.pp.log1p(_tmp_cc)
    sc.tl.score_genes_cell_cycle(_tmp_cc, s_genes=s_genes_cc, g2m_genes=g2m_genes_cc)
    adata.obs["S_score"] = _tmp_cc.obs["S_score"]
    adata.obs["G2M_score"] = _tmp_cc.obs["G2M_score"]
    adata.obs["phase"] = _tmp_cc.obs["phase"]
    del _tmp_cc
else:
    sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes_cc, g2m_genes=g2m_genes_cc)
print(f"细胞周期评分完成 → obs 新增: S_score, G2M_score, phase")
_phase_counts = adata.obs["phase"].value_counts()
for phase, count in _phase_counts.items():
    print(f"  {phase}: {count} ({100*count/adata.n_obs:.1f}%)")

# 可选：从 HVG 排除细胞周期基因（默认 False）
# 当增殖信号过度主导 PCA 时启用——优先用 stage4 covariate 消除周期效应
if EXCLUDE_CELL_CYCLE_FROM_HVG:
    cc_genes = set(s_genes_cc + g2m_genes_cc)
    cc_mask = adata.var_names.isin(cc_genes)
    excluded_counts["CellCycle"] = int((adata.var["highly_variable"] & cc_mask).sum())
    exclude_mask |= cc_mask

# 移除
n_before = int(adata.var["highly_variable"].sum())
adata.var.loc[exclude_mask, "highly_variable"] = False
n_after = int(adata.var["highly_variable"].sum())
n_removed = n_before - n_after

# 排除后数量不足警告
_requested = _n_genes_values[-1] if isinstance(N_TOP_GENES, list) else N_TOP_GENES
if n_after < _requested * 0.8:
    print(f"⚠️ HVG 排除后仅剩 {n_after}（< 80% of requested {_requested}），建议增大 N_TOP_GENES 或减少排除类别")

print(f"HVG 排除列表：移除 {n_removed} 基因")
for cat, count in excluded_counts.items():
    if count > 0:
        print(f"  {cat}: {count}")
print(f"剩余 HVG: {n_after}")
print(f"（不做补位——如需精确数量请增大 N_TOP_GENES 初始值）")

## 强制纳入关键基因

`FORCED_INCLUDE_GENES` 允许 PI 将特定关键基因强制加入 HVG 列表，
即使这些基因在统计上未达到高可变阈值。

**为什么需要强制纳入？**
研究者关心的关键生物学 axis（如炎-癌转化 marker：
`CDX2`、`TFF3`、`GKN1`、`MUC2`、`OLFM4`、`LGR5` 等）
可能在当前数据集中的表达变异度未排进前 N——
但这不代表它们不重要。强制纳入保证它们一定进入下游 PCA 空间，
从而在聚类、UMAP 可视化和细胞类型鉴定中被利用。

加 20-30 个关键基因不会实质性改变 HVG 整体空间结构，
但能确保研究者关注的生物学信号不被遗漏。


In [ ]:
# === 强制纳入关键基因 ===
# 在 HVG 排除逻辑之后执行——PI 指定的关键基因强制设为 highly_variable=True
if FORCED_INCLUDE_GENES:
    found = [g for g in FORCED_INCLUDE_GENES if g in adata.var_names]
    not_found = [g for g in FORCED_INCLUDE_GENES if g not in adata.var_names]
    n_before = int(adata.var["highly_variable"].sum())
    adata.var.loc[found, "highly_variable"] = True
    n_after = int(adata.var["highly_variable"].sum())
    newly_added = n_after - n_before
    print(f"强制纳入: {len(found)} 基因找到, {newly_added} 新增到 HVG (总 HVG: {n_after})")
    if not_found:
        print(f"  ⚠️ 未找到（检查基因名大小写）: {not_found}")
else:
    print("FORCED_INCLUDE_GENES 为空，跳过强制纳入")


In [ ]:
# HVG 组成分析：选中的 HVG 是什么类别的基因？
# 如果某类别占比异常高 → 需要调整排除列表或 N_TOP_GENES
hvg_genes = adata.var_names[adata.var["highly_variable"]]
n_hvg = len(hvg_genes)

categories = {
    "MT（线粒体）": hvg_genes.str.upper().str.startswith("MT-").sum(),
    "Ribo（核糖体）": hvg_genes.str.upper().str.match("^(RPS|RPL)").sum(),
    "HB（血红蛋白）": hvg_genes.str.upper().str.match("^HB[^P]").sum(),
    "IG（免疫球蛋白）": hvg_genes.str.upper().str.match("^IG[HKL]").sum(),
}

# Cell Cycle（Tirosh 2015 S + G2M 期 marker genes）
# 若 EXCLUDE_CELL_CYCLE_FROM_HVG=False，仅统计不排除，供 PI 判断是否需要排除
_s = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
_g2m = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
categories["Cell Cycle（细胞周期）"] = hvg_genes.isin(set(_s + _g2m)).sum()

categories["其他"] = n_hvg - sum(categories.values())

print(f"===== HVG 组成分析（{n_hvg} genes）=====")
for cat, count in categories.items():
    pct = 100 * count / n_hvg if n_hvg > 0 else 0
    flag = " ⚠️" if pct > 5 and cat != "其他" else ""
    print(f"  {cat}: {count} ({pct:.1f}%){flag}")

# 如果 IG 基因占比 > 5%，说明 B 细胞过于突出
if categories.get("IG（免疫球蛋白）", 0) / max(n_hvg, 1) > 0.05:
    print("\n⚠️ IG 基因占比 > 5%：B 细胞信号可能主导聚类")
    print("  → 考虑将 IG 基因加入 CUSTOM_EXCLUDE_PATTERNS: [\"^IG[HKL]\"]")

In [ ]:
# === Marker 库覆盖率诊断 ===
# 检查 gastric_TEST_markers.csv 中的 marker 基因有多少被当前 HVG 选中
# 未被选中的 marker 不会参与下游聚类——影响对应细胞类型的鉴定
_marker_path = os.path.join("references", "markers", "gastric_TEST_markers.csv")
if os.path.exists(_marker_path):
    _marker_df = pd.read_csv(_marker_path, comment="#")
    # 兼容列名：尝试 gene / gene_symbol / Gene / marker
    _gene_col = None
    for col in ["gene", "gene_symbol", "Gene", "marker"]:
        if col in _marker_df.columns:
            _gene_col = col
            break
    if _gene_col:
        _marker_genes = _marker_df[_gene_col].dropna().unique().tolist()
        _in_var = [g for g in _marker_genes if g in adata.var_names]
        _in_hvg = [g for g in _in_var if adata.var.loc[g, "highly_variable"]]
        _not_in_hvg = [g for g in _in_var if not adata.var.loc[g, "highly_variable"]]
        print(f"\n===== Marker 库覆盖率（{_marker_path}）=====")
        print(f"  Marker 总基因: {len(_marker_genes)}")
        print(f"  在 adata 中: {len(_in_var)}")
        print(f"  在 HVG 中: {len(_in_hvg)} ({100*len(_in_hvg)/max(len(_in_var),1):.1f}%)")
        if _not_in_hvg:
            print(f"  未被选入 HVG: {_not_in_hvg[:20]}")  # 最多显示 20 个
            print(f"  → 如需这些基因参与聚类，加入 FORCED_INCLUDE_GENES")
    else:
        print(f"Marker CSV 列名不匹配，期望 gene/gene_symbol/Gene/marker 之一")
else:
    print(f"未找到 marker 文件 {_marker_path}，跳过覆盖率检查")


In [ ]:
# === 胃粘膜关键谱系基因 HVG 覆盖诊断 ===
# GCPL 炎-癌转化研究中，这些谱系特异性基因必须参与聚类——
# 如果某 lineage 的核心 marker 未被 HVG 选中，下游 UMAP/PCA 将无法分辨该细胞群体
_GASTRIC_LINEAGE_PANEL = {
    "壁细胞": ["ATP4A", "ATP4B", "GIF"],
    "主细胞": ["PGA3", "PGA4", "PGA5", "LIPF", "PGC"],
    "表面黏液": ["MUC5AC", "TFF1", "GKN1", "GKN2"],
    "颈黏液": ["MUC6", "TFF2"],
    "SPEM": ["WFDC2", "CD44", "AQP5"],
    "肠化": ["CDX2", "MUC2", "TFF3", "VIL1", "OLFM4"],
    "内分泌": ["CHGA", "CHGB", "SYP"],
}

print("\n===== 胃粘膜关键谱系基因 HVG 覆盖 =====\n")
_total_panel = 0
_total_in_hvg = 0
_missing_critical = []

for lineage, genes in _GASTRIC_LINEAGE_PANEL.items():
    _in_var = [g for g in genes if g in adata.var_names]
    _in_hvg = [g for g in _in_var if adata.var.loc[g, "highly_variable"]]
    _not_hvg = [g for g in _in_var if not adata.var.loc[g, "highly_variable"]]
    _total_panel += len(_in_var)
    _total_in_hvg += len(_in_hvg)

    _status = "✓" if len(_not_hvg) == 0 else "⚠️"
    print(f"  {_status} {lineage}: {len(_in_hvg)}/{len(_in_var)} 在 HVG 中", end="")
    if _not_hvg:
        print(f"  (未覆盖: {_not_hvg})")
        _missing_critical.extend(_not_hvg)
    else:
        print()

_coverage_pct = 100 * _total_in_hvg / max(_total_panel, 1)
print(f"\n  总覆盖率: {_total_in_hvg}/{_total_panel} ({_coverage_pct:.0f}%)")

if _missing_critical:
    print(f"\n  → {len(_missing_critical)} 个关键谱系基因未进入 HVG:")
    print(f"    {_missing_critical}")
    print(f"    建议加入 FORCED_INCLUDE_GENES 参数确保聚类可见")
else:
    print(f"\n  ✓ 所有关键谱系基因均在 HVG 中——GCPL 分析就绪")

## HVG 诊断图

**平均表达量 vs 离散度图**：每个点是一个基因。X 轴是平均表达量（log scale），
Y 轴是标准化离散度。蓝色点为选中的 HVG，灰色点为未选中。
理想情况下蓝色点应在各个表达水平上均匀覆盖高离散度区域。

**跨批次出现频率图**（batch-aware 模式专属）：展示 HVG 在多少个 batch 中被选中。
出现频率越高的基因越可能是跨数据集保守的真信号，而非单数据集的技术噪声。

In [ ]:
# === HVG 诊断图 ===
# 图1：平均表达量 vs 标准化离散度（标准 scanpy 诊断图）
sc.pl.highly_variable_genes(adata, show=True)
plt.savefig("results/figures/03_hvg_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()
print("HVG 诊断图已保存至 results/figures/03_hvg_diagnostic.png")

# 图2：batch-aware 时额外输出——per-batch HVG 出现频率分布
if BATCH_AWARE_HVG and "highly_variable_nbatches" in adata.var.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    nbatches_dist = adata.var["highly_variable_nbatches"].value_counts().sort_index()
    nbatches_dist.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_xlabel("出现在几个 batch 中")
    ax.set_ylabel("基因数")
    ax.set_title("HVG 跨 batch 出现频率\n（越多 batch 共享 = 越可能是真信号）")
    plt.tight_layout()
    plt.savefig("results/figures/03_hvg_nbatches.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    # 统计摘要
    n_total_batches = adata.obs[HVG_BATCH_KEY].nunique()
    n_ubiquitous = int((adata.var["highly_variable_nbatches"] == n_total_batches).sum())
    print(f"总 batch 数: {n_total_batches}")
    print(f"在所有 {n_total_batches} 个 batch 中均为 HVG 的基因: {n_ubiquitous}")
elif BATCH_AWARE_HVG:
    print("（batch-aware 模式下未检测到 highly_variable_nbatches 列，可能 flavor 不支持此输出）")

## （可选）回归混杂变量

`sc.pp.regress_out` 用线性回归从表达矩阵中移除指定协变量的影响。

**为什么不推荐在此阶段回归？**
1. 回归会 **densify** 稀疏矩阵，内存占用可能膨胀 10-50 倍
2. 下游 Harmony/scVI 已内置批次/协变量处理，通常无需提前回归
3. 线性回归假设协变量与表达量是线性关系——对单细胞 counts 数据往往不成立

**什么时候启用？** PI 在 04 嵌入后发现某个混杂因素（如 MT% 或 total_counts）
严重主导了 PCA/UMAP 且 Harmony/scVI 无法充分校正时，才考虑回归。

In [ ]:
# === 回归混杂变量（可选）===
if REGRESS_OUT:
    print(f"⚠ 回归混杂变量：{REGRESS_OUT}")
    print("  注意：回归会 densify 矩阵，大幅增加内存。通常不推荐——")
    print("  更好的做法是在 04 嵌入中通过 batch_key/covariates 处理。")

    # 检查所有回归变量是否存在于 adata.obs
    missing = [v for v in REGRESS_OUT if v not in adata.obs.columns]
    if missing:
        raise KeyError(f"REGRESS_OUT 中包含不存在于 adata.obs 的列: {missing}。"
                       f"可用列: {list(adata.obs.columns)}")

    # NaN 值防御：某些变量可能因上游跳过而含 NaN
    _valid_regress = []
    for var in REGRESS_OUT:
        if var not in adata.obs.columns:
            print(f"  ⚠️ 跳过 '{var}'：不在 obs 中")
            continue
        nan_pct = adata.obs[var].isna().mean()
        if nan_pct > 0.05:
            print(f"  ⚠️ 跳过 '{var}'：{nan_pct:.1%} NaN（> 5% 阈值）")
            continue
        elif nan_pct > 0:
            # 少量 NaN：填充中位数
            adata.obs[var] = adata.obs[var].fillna(adata.obs[var].median())
            print(f"  '{var}': {nan_pct:.1%} NaN 已填充为中位数")
        _valid_regress.append(var)

    if _valid_regress:
        # 回归前的内存快照
        was_sparse = sp.issparse(adata.X)
        sc.pp.regress_out(adata, _valid_regress)
        adata.X = adata.X.astype(np.float32)

        # 尝试恢复 sparse（如果足够稀疏）
        if not sp.issparse(adata.X):
            zero_fraction = (adata.X == 0).mean()
            if zero_fraction > 0.5:
                adata.X = sp.csr_matrix(adata.X)
                print(f"  回归后恢复 sparse（稀疏度 {zero_fraction:.1%}）")
            else:
                print(f"  ⚠ 回归后矩阵不够稀疏（零值占比 {zero_fraction:.1%}），保持 dense")
                print(f"    内存占用约 {adata.X.nbytes / 1024**2:.0f} MB")
    else:
        print("  所有回归变量均不可用或含过多 NaN，跳过回归")
else:
    print("✓ 跳过回归（REGRESS_OUT 为空）")

## （可选）缩放

`sc.pp.scale` 将每个基因的表达量标准化到均值为 0、方差为 1——
使 PCA 中各基因权重均等，避免高表达基因主导主成分。

**注意**：
- scale 会 **densify** 矩阵
- **Harmony/scVI 不需要 scale**——Harmony 在 PCA 空间校正、scVI 用原始 counts + 内部归一化
- PCA-only 管线（不做任何批次校正的简单分析）可能需要
- scale 前会保存 normalized 层到 `adata.layers["normalized"]`，以便后续可恢复

In [ ]:
# === 缩放（可选）===
if SCALE:
    print(f"缩放：max_value={MAX_SCALE_VALUE}")
    print("  注意：scale 会 densify 矩阵。Harmony/scVI 不需要 scale。")

    # 保存 normalized 到 layer 以便后续可恢复 sparse
    adata.layers["normalized"] = adata.X.copy()
    sc.pp.scale(adata, max_value=MAX_SCALE_VALUE)
    adata.X = adata.X.astype(np.float32)
    print(f"  缩放后 X mean={adata.X.mean():.4f}, std={adata.X.std():.4f}")
else:
    print("✓ 跳过缩放（SCALE=False，Harmony/scVI 不需要）")

## Sanity PCA 诊断（轻量级，不修改 adata）

在写入 checkpoint 前执行一次轻量级 PCA（仅 5 个主成分），
帮助 PI 快速判断标准化是否干净、batch 效应有多强。

**诊断项**：
- **PC1 vs library size**：标准化后 PC1 仍与文库大小相关 → 标准化不够彻底
- **PC1 ANOVA by batch**：PC1 在各 batch 间差异显著 → batch effect 需要重点处理
- **方差解释比**：PC1 占比 > 30% → 可能存在强混杂因素

**注意**：此 PCA 使用临时 `adata` 子集，不修改 `adata.obsm`/`adata.uns`，
不影响后续 stage4 的正式 PCA。


### 诊断标准与行动

**正常**（继续运行）：
- PC1 vs library size 相关系数 r < 0.3
- ANOVA p > 1e-10（batch 不主导 PC1）

**异常**（需处理）：
- **r > 0.3 或 p < 1e-10**：该批次与 PC 强关联，批次效应未消除
  - 立即行动：回到 02_merged 检查 BATCH_KEY 是否正确，或在本 stage 考虑 `sc.pp.combat()` 批次校正
  - 如果是已知的生物学差异（如不同组织来源本该不同），可继续，但需在结果解读时注意

**轻度关联**（0.2 < r < 0.3 或接近阈值）：
- 查看该批次细胞在 PCA plot 中的分布
- 如果在主要 PC 上分离明显，考虑批次校正
- 否则可继续，在 stage4 使用 Harmony/scVI 校正


In [ ]:
# === Sanity PCA 诊断（轻量级，不修改 adata）===
print("\n===== 标准化效果快速诊断 =====")
_hvg_mask = adata.var["highly_variable"]
_tmp_pca = adata[:, _hvg_mask].copy()

# scale + PCA（临时对象，不影响 adata）
sc.pp.scale(_tmp_pca, max_value=10)
sc.tl.pca(_tmp_pca, n_comps=N_PCS_SANITY, random_state=RANDOM_SEED)

# Elbow plot：帮助 PI 确定最终分析用的 PC 数
sc.pl.pca_variance_ratio(_tmp_pca, n_pcs=N_PCS_SANITY, log=True)
plt.title(f"Elbow Plot（Sanity PCA，{N_PCS_SANITY} PC）\n肘部（斜率显著变缓处）即最优 PC 数")
plt.savefig("results/figures/03_pca_elbow.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Elbow plot 已保存：results/figures/03_pca_elbow.png")

# PC1 vs library size
_log_lib = np.log1p(np.array(adata.layers["counts"].sum(axis=1)).flatten())
_pc1 = _tmp_pca.obsm["X_pca"][:, 0]
_r_lib = np.corrcoef(_log_lib, _pc1)[0, 1]
print(f"  PC1 vs log(total_counts): r = {_r_lib:.3f}", end="")
if abs(_r_lib) > 0.3:
    print(" ⚠️ 标准化后 PC1 仍受 library size 驱动")
else:
    print(" ✓ library size 效应已消除")

# PC1 vs batch (ANOVA)
if HVG_BATCH_KEY in adata.obs.columns and adata.obs[HVG_BATCH_KEY].nunique() > 1:
    from scipy.stats import f_oneway
    _groups = [_pc1[adata.obs[HVG_BATCH_KEY] == b] for b in adata.obs[HVG_BATCH_KEY].unique()]
    _f_stat, _p_val = f_oneway(*_groups)
    print(f"  PC1 ANOVA by {HVG_BATCH_KEY}: F={_f_stat:.1f}, p={_p_val:.2e}", end="")
    if _p_val < 1e-10:
        print(" → batch effect 显著，stage4 整合方法需重点处理")
    else:
        print(" ✓ batch 不主导 PC1")

# 方差解释比
_var_ratio = _tmp_pca.uns["pca"]["variance_ratio"]
print(f"  PCA 方差解释: PC1={_var_ratio[0]:.3f}, PC2={_var_ratio[1]:.3f}, PC3={_var_ratio[2]:.3f}")
if _var_ratio[0] > 0.3:
    print("  ⚠️ PC1 方差占比 > 30%——可能存在强混杂因素，检查 batch/library_size/MT%")

del _tmp_pca
gc.collect()


## 决策点：确定最终 PC 数

查看上方 elbow plot，在曲线"肘部"（斜率显著变缓处）选择 PC 数：

- **典型范围**：15–30 PC
- **判断标准**：肘部之后的 PC 主要是噪声，方差贡献趋于平稳
- **行动**：回到 PARAMS cell，将 `N_PCS_FINAL` 设为选择的值，然后继续运行

如果 `N_PCS_FINAL` 仍为 None，下方 checkpoint cell 会报错提醒，不会写入 checkpoint。

## 参数记录 + Checkpoint

将标准化参数写入 `adata.uns` 以便追溯，执行内存自检确保数据完整性，
然后将产出写入磁盘形成 03 checkpoint。

**写入的 uns 字段**：
- `normalize_v1` — 本次运行的完整参数记录
- `stage` / `status` / `upstream` / `version` — 管线追溯链

**内存自检**：确认 `adata.X` 是 float32（sparse 或 dense 均可，
因 regress_out 或 scale 可能导致 dense——允许但警告）。

### Stage 03 Verdict

本 stage 完成后应确认：
- [ ] PC1 vs library size 相关 < 0.3（标准化有效）
- [ ] HVG 覆盖关键胃粘膜谱系基因（见上方诊断）
- [ ] 细胞周期评分已存入 obs（供 04 使用）


In [ ]:
# === N_PCS_FINAL 校验 ===
# 确保 PI 已根据 elbow plot 填入最终 PC 数，防止使用 None 写入 checkpoint
if N_PCS_FINAL is None:
    raise ValueError(
        "N_PCS_FINAL 未设置。请先查看上方 elbow plot，"
        "在 PARAMS cell 中将 N_PCS_FINAL 设为选择的 PC 数（通常 15-30），"
        "然后重新运行此 cell。"
    )
print(f"N_PCS_FINAL = {N_PCS_FINAL}  ✓")

# === 参数记录 + Checkpoint ===

# 记录运行参数
# 记录实际使用的值，而非参数列表末尾值
_actual_n_top_genes = int(adata.var["highly_variable"].sum())  # 实际选入的 HVG 数
# _flavor_values 在 03-hvg-sweep 中由 Pearson residuals 兼容逻辑可能已修改，取最终用到的 flavor
_actual_hvg_flavor = _flavor_values[-1] if isinstance(HVG_FLAVOR, list) else HVG_FLAVOR

adata.uns["normalize_v1"] = {
    "method": NORMALIZATION_METHOD,
    "target_sum": TARGET_SUM if NORMALIZATION_METHOD == "standard" else None,
    "n_top_genes": _actual_n_top_genes,
    "hvg_flavor": _actual_hvg_flavor,
    "batch_aware": BATCH_AWARE_HVG,
    "hvg_batch_key": HVG_BATCH_KEY if BATCH_AWARE_HVG else None,
    "excluded_from_hvg": excluded_counts,
    "forced_include_genes": FORCED_INCLUDE_GENES,
    "regress_out": REGRESS_OUT,
    "regress_out_nan_fill": "median" if REGRESS_OUT else None,
    "scale": SCALE,
}
print("normalize_v1:", adata.uns["normalize_v1"])

# === expression_contract：记录 X 当前尺度与 counts 来源（决策 1/2 schema）===
# processing_history 追加当前步骤，随 NORMALIZATION_METHOD 变化
if NORMALIZATION_METHOD == "standard":
    _processing_step = f"normalize_total(target_sum={TARGET_SUM}) + log1p"
elif NORMALIZATION_METHOD == "pearson_residuals":
    _processing_step = "normalize_pearson_residuals"
else:
    _processing_step = f"normalize_{NORMALIZATION_METHOD}"

_upstream_history = _upstream_contract.get("processing_history", [])
# 动态确定 x_scale：pearson_residuals 输出残差空间，非 log1p 空间
x_scale = "pearson_residuals" if NORMALIZATION_METHOD == "pearson_residuals" else "normalized_log1p"
adata.uns["expression_contract"] = {
    "x_scale": x_scale,
    "counts_layer": "counts",
    "counts_source": _upstream_contract.get("counts_source", "layers[counts]"),
    "counts_validated": _upstream_contract.get("counts_validated", True),
    "counts_integer_check": _upstream_contract.get("counts_integer_check", "full"),
    "soupx_layer": None,
    "processing_history": _upstream_history + [_processing_step],
    "stage": "03",
}
# 自验证：确保契约字段符合 schema
validate_expression_contract(adata, expected_scale=x_scale, stage="03")
print("expression_contract:", adata.uns["expression_contract"])

# 所有可失败的门禁、参数快照与运行溯源先完成，再占用 RUN_ID。
_x_values = adata.X.data if sp.issparse(adata.X) else np.asarray(adata.X)
_hvg_count = int(adata.var["highly_variable"].sum()) if "highly_variable" in adata.var else 0
hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "x_finite": bool(np.isfinite(_x_values).all()),
    "x_float32": adata.X.dtype == np.float32,
    "hvg_non_empty": _hvg_count > 0,
    "counts_layer_preserved": _counts_integrity_checked,
}

# numpy.bool_ 不是 Python bool 的子类，json.dump 不认它；此处统一转换，
# 否则 checkpoint 写出时会抛 TypeError: Object of type bool_ is not JSON serializable
hard_postconditions = {
    k: v.item() if hasattr(v, 'item') and callable(v.item) else v
    for k, v in hard_postconditions.items()
}

stage_status = determine_stage_status({}, hard_postconditions, allow_no_required_methods=True)
effective_parameters = snapshot_effective_parameters(
    globals(), exclude=("UPSTREAM_CHECKPOINT",), path_root=Path(_root)
)
runtime_provenance = collect_runtime_provenance(
    _root, ("anndata", "scanpy", "numpy", "pandas", "scipy")
)
run_paths = prepare_run(RUN_ROOT, RUN_ID)
manifest_payload = {
    "run_id": run_paths.run_id, "stage": "03_normalized", "stage_status": stage_status.value,
    "inputs": [upstream_input], "effective_parameters": effective_parameters,
    "runtime_provenance": runtime_provenance, "hard_postconditions": hard_postconditions,
}
if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 03 FAILED: {hard_postconditions}")
adata.uns["stage"] = "03_normalized"
adata.uns["status"] = stage_status.value
adata.uns["upstream"] = [str(UPSTREAM_CHECKPOINT)]
adata.uns["upstream_inputs"] = [upstream_input]
adata.uns["version"] = f"v{OUTPUT_VERSION}"
adata.uns["run_id"] = run_paths.run_id
draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise
manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}
atomic_write_json(run_paths.manifest_path, manifest_payload)
OUTPUT_PATH = str(promote_run(run_paths))
print(f"✓ 提升 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes, HVG={_hvg_count})")
del adata
gc.collect()